# Modélisation vol + météo

## 1. Contexte et objectif du notebook

Les trois sources sont dans PostgreSQL (étoile).

Premier modèle : vols + météo + actualités jointes **par date UTC de `first_seen`**.

Cible = **durée observée** (`duree_minutes`), pas un retard.

Le score BERT est dans le notebook 05 et `docs/experience_actualites_bert.md`. Ici : diagnostic cible / split / LightGBM (`python -m src.models.entrainer`).

Les cellules code relisent **la base locale** : les COUNT changent si l'entrepôt a bougé. Les commentaires chiffrés plus bas sont un **run CLI du 01/09/2026**. Le livrable de référence est le **Rendu 3 (04/09)** : 21 054 vols, 3 664 lignes modèle, LightGBM **54,02** vs médiane **44,07**.


In [1]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import pandas as pd

RACINE_PROJET = os.path.abspath(os.path.join(os.getcwd(), ".."))
if RACINE_PROJET not in sys.path:
    sys.path.insert(0, RACINE_PROJET)
os.chdir(RACINE_PROJET)

from src.models.entrainer import executer_pipeline
from src.models.jeu import (
    COLONNES_FEATURES,
    COLONNES_INTERDITES_FEATURES,
    NOM_CIBLE,
    ajouter_cible_duree,
    charger_vols_meteo,
    construire_features,
    diagnostiquer_volumetrie,
    exclure_durees_aberrantes,
    filtrer_jeu_modele,
    retenir_cible,
    split_temporel,
)

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", message="Pandas requires version")

C:\Users\jeann\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 2. Chargement depuis PostgreSQL

Je lis `fact_flights` joint à `dim_date`, `dim_weather` et `dim_news_sentiment` (date UTC de `first_seen`) via `charger_vols_meteo`. Aucun appel HTTP. Si l'instance Postgres n'est pas joignable, la fonction échoue.

In [2]:
df = charger_vols_meteo()
print(f"Lignes : {len(df)}")
print(f"Colonnes : {len(df.columns)}")
print(f"first_seen min / max : {df['first_seen'].min()} / {df['first_seen'].max()}")
df.head(3)

2026-08-28 14:28:57,400 - src.models.jeu - INFO - Jeu brut chargé : 1286 ligne(s), 24 colonne(s).


Lignes : 1286
Colonnes : 24
first_seen min / max : 2026-08-25 13:20:36+00:00 / 2026-08-28 12:25:15+00:00


,flight_id,icao24,callsign,first_seen,last_seen,est_departure_airport,est_arrival_airport,date_id,retard_minutes,weather_origin_id,weather_dest_id,jour_semaine,mois,temperature_origine,vent_origine,rafale_origine,pluie_origine,visibilite_origine,humidite_origine,temperature_dest,vent_dest,rafale_dest,pluie_dest,visibilite_dest
0,753,344416,IBS1527,2026-08-25 19:33:29+00:00,2026-08-25 21:43:31+00:00,LEMD,None,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,754,4ca2bf,RYR1GR,2026-08-25 19:21:15+00:00,2026-08-25 21:10:28+00:00,LEMD,LIAU,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,755,344341,AEA9172,2026-08-25 19:20:20+00:00,2026-08-25 21:41:47+00:00,LEMD,None,20260825,NaN,NaN,NaN,2,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Volumétrie, météo, proxy `retard_minutes`

Je passe le DataFrame à `diagnostiquer_volumetrie` : totaux, part de météo origine / destination, NULL sur le proxy `retard_minutes` (durée − médiane de route, **pas** un horaire), routes assez peuplées.


In [3]:
diag = diagnostiquer_volumetrie(df)
for cle, valeur in diag.items():
    print(f"{cle}: {valeur}")

nb_vols: 1286
retard_non_null: 11
pct_retard_null: 99.1
weather_origin: 711
pct_weather_origin: 55.3
weather_dest: 12
pct_weather_dest: 0.9
dest_renseignee: 308
last_seen_ok: 1286
meteo_et_duree: 711
routes_ge5: 3
vols_sur_routes_ge5: 17
routes_meteo_ge5: 0
vols_meteo_sur_routes_ge5: 0
first_seen_min: 2026-08-25T13:20:36+00:00
first_seen_max: 2026-08-28T12:25:15+00:00
nb_jours: 4


**Observations.** Les sorties code sont l'état **actuel** de la base. Les chiffres commentés ici sont ceux du CLI **01/09** (7 524 vols, ~18 % avec météo origine). Référence officielle : **Rendu 3, 04/09** (21 054 vols, 3 664 lignes modèle). `retard_minutes` = proxy, pas un slot.


## 4. Choix de cible

Trois options :

- **A** : classer « plus long que la médiane historique du couple OD ».
- **B** : régresser la **durée observée** `last_seen − first_seen`.
- **C** : régresser le proxy `retard_minutes` (écart à la médiane de route).


In [4]:
code_cible = retenir_cible(diag)
print(f"Cible retenue : {code_cible}")
print(f"Nom de colonne : {NOM_CIBLE}")
print(f"retard_minutes non NULL : {diag['retard_non_null']}")
print(f"retard_minutes non NULL avec météo origine : {diag['retard_non_null_avec_meteo']}")
print(f"vols sur routes OD >= 5 avec météo origine : {diag['vols_meteo_sur_routes_ge5']}")
print(f"vols avec météo origine et durée : {diag['meteo_et_duree']}")

Cible retenue : B
Nom de colonne : duree_minutes
retard_minutes non NULL : 11
vols sur routes OD >= 5 avec météo origine : 0
vols avec météo origine et durée : 711


**Observations.** Le diagnostic retient **B**. Sur le run 01/09, A et C n'avaient pas assez de lignes avec météo origine. Ce n'est pas une prédiction de retard DGAC (réel − programmé) : les trois APIs n'exposent pas de slot.


## 5. Features et fuite de données

Je filtre les lignes sans météo origine, je calcule `duree_minutes`, j'écarte les durées > 8 h, puis `construire_features` assemble X (météo +, par défaut, `nb_articles_actu` et `proxy_perturbation_actu`). `last_seen` sert à la cible ; il ne doit pas se retrouver dans X. Les actualités se joignent par date UTC de `first_seen`, pas par vol.

In [5]:
jeu = filtrer_jeu_modele(df)
jeu = ajouter_cible_duree(jeu)
jeu = exclure_durees_aberrantes(jeu)
print(f"Lignes après filtre météo origine et plafond 8 h : {len(jeu)}")
print(jeu[NOM_CIBLE].describe().round(2))

X, y = construire_features(jeu)
print("Colonnes de X :", list(X.columns))
print("Intersection avec colonnes interdites :", set(X.columns) & COLONNES_INTERDITES_FEATURES)
print("last_seen dans X :", "last_seen" in X.columns)
print("cible dans X :", NOM_CIBLE in X.columns)
X.head(3)

2026-08-28 14:28:57,510 - src.models.jeu - INFO - Jeu modèle : 711 ligne(s) avec météo origine et durée observée.


Lignes après filtre météo origine : 711
count     711.00
mean       69.85
std        80.97
min         0.32
25%        29.64
50%        53.97
75%        89.97
max      1412.65
Name: duree_minutes, dtype: float64
Colonnes de X : ['hub_origine', 'jour_semaine', 'mois', 'heure_utc', 'temperature_origine', 'vent_origine', 'rafale_origine', 'pluie_origine', 'visibilite_origine', 'humidite_origine', 'temperature_dest', 'vent_dest', 'rafale_dest', 'pluie_dest', 'visibilite_dest']
Intersection avec colonnes interdites : set()
last_seen dans X : False
cible dans X : False


,hub_origine,jour_semaine,mois,heure_utc,temperature_origine,vent_origine,rafale_origine,pluie_origine,visibilite_origine,humidite_origine,temperature_dest,vent_dest,rafale_dest,pluie_dest,visibilite_dest
15,LEMD,2,8,17.433333,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN
16,LEMD,2,8,17.116667,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN
17,LEMD,2,8,16.900000,27.63,6.69,0.0,0.0,10000.0,38.0,NaN,NaN,NaN,NaN,NaN


In [6]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(y, bins=40, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Durée observée (minutes)")
ax.set_ylabel("Nombre de vols")
ax.set_title("Distribution de duree_minutes (jeu avec météo origine)")
plt.tight_layout()
plt.show()

C:\Users\jeann\AppData\Local\Temp\ipykernel_13836\2508720148.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**Observations.** Run 01/09 : ~1 329 lignes après plafond 8 h. X = hub, calendrier (heure UTC de `first_seen`), météo origine, météo dest souvent vide, `nb_articles_actu` / `proxy_perturbation_actu`. Pas de `last_seen` dans X, pas le proxy comme feature. Le test de ce run n'avait **aucun** article. Référence officielle des volumes : Rendu 3 (04/09).


## 6. Split temporel

Je coupe sur `first_seen` au quantile 70 % : tout ce qui est **avant** le seuil va au train, tout ce qui est **après** au test. Pas de tirage aléatoire i.i.d.

In [7]:
train, test = split_temporel(jeu, fraction_train=0.70)
print(f"Train : {len(train)} vols, jusqu'à {train['first_seen'].max()}")
print(f"Test  : {len(test)} vols, depuis {test['first_seen'].min()}")
print(
    "max(train) < min(test) :",
    train["first_seen"].max() < test["first_seen"].min(),
)

2026-08-28 14:28:57,793 - src.models.jeu - INFO - Split temporel : train=497 (jusqu'à 2026-08-26T14:43:31+00:00), test=214 (depuis 2026-08-26T14:43:43+00:00).


Train : 497 vols, jusqu'à 2026-08-26 14:43:31+00:00
Test  : 214 vols, depuis 2026-08-26 14:43:43+00:00
max(train) < min(test) : True


**Observations.** Split temporel 70/30 sur `first_seen` (pas de k-fold : ça mélangerait passé et futur). Run 01/09 : train avec actu, test du 01/09 **sans** article. Les effectifs train/test du Rendu 3 (04/09) sont 2 453 / 1 052.


## 7. Baseline médiane et LightGBM

Même enchaînement que `python -m src.models.entrainer` : `executer_pipeline` sur le DataFrame déjà chargé (pas de second aller-retour SQL). Deux baselines : médiane globale du train, et médiane **par heure UTC** (repli sur la globale si l'heure a moins de 5 vols en train). Pas de sauvegarde `joblib` ici (le script CLI l'écrit dans `models/`).

In [8]:
rapport = executer_pipeline(jeu=df, sauvegarder=False)
print(f"Cible : {rapport['cible']} (code {rapport['code_cible']})")
print(f"Train / test : {rapport['n_train']} / {rapport['n_test']}")
print(f"Fin train : {rapport['fin_train']}")
print(f"Début test : {rapport['debut_test']}")
print("LightGBM :", {k: round(v, 2) for k, v in rapport['metriques_modele'].items()})
print("Médiane globale :", {k: round(v, 2) for k, v in rapport['metriques_baseline'].items()})
print("Médiane horaire :", {k: round(v, 2) for k, v in rapport['metriques_baseline_horaire'].items()})
print(f"Bat la globale : {rapport['bat_la_baseline']}")
print(f"Bat l'horaire : {rapport['bat_la_baseline_horaire']}")

2026-08-28 14:28:57,830 - src.models.jeu - INFO - Jeu modèle : 711 ligne(s) avec météo origine et durée observée.
2026-08-28 14:28:57,844 - src.models.jeu - INFO - Split temporel : train=497 (jusqu'à 2026-08-26T14:43:31+00:00), test=214 (depuis 2026-08-26T14:43:43+00:00).
2026-08-28 14:28:58,172 - src.models.entrainer - INFO - Cible duree_minutes (B). Train=497, test=214. LightGBM RMSE=74.00 MAE=66.52. Baseline médiane RMSE=43.77 MAE=37.94. Bat la baseline : False.


Cible : duree_minutes (code B)
Train / test : 497 / 214
Fin train : 2026-08-26T14:43:31+00:00
Début test : 2026-08-26T14:43:43+00:00
LightGBM : {'rmse': 74.0, 'mae': 66.52}
Baseline : {'rmse': 43.77, 'mae': 37.94}
Bat la baseline : False


**Observations.** Les sorties de la cellule peuvent être un ancien instantané. CLI **01/09** : LightGBM 55,22 vs médiane 40,43 (horaire 69,60) ; actu −1,5 RMSE, test sans article, pas de BERT.

**Référence Rendu 3 (04/09)** : LightGBM **54,02** vs médiane **44,07** (horaire 47,99). Dans les deux cas LightGBM perd contre la médiane globale.


## 8. Quelles variables ont servi ?

In [9]:
importances = pd.Series(rapport["importance_features"]).sort_values(ascending=True)
print(importances.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind="barh", ax=ax, color="#4C72B0")
ax.set_xlabel("Importance LightGBM (splits)")
ax.set_title("Importance des features")
plt.tight_layout()
plt.show()

mois                     0
rafale_origine           0
pluie_origine            0
visibilite_origine       0
temperature_dest         0
vent_dest                0
rafale_dest              0
pluie_dest               0
visibilite_dest          0
hub_origine              9
jour_semaine            88
humidite_origine       105
temperature_origine    152
vent_origine           170
heure_utc              799


C:\Users\jeann\AppData\Local\Temp\ipykernel_13836\799537175.py:9: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


**Observations.** L'heure UTC domine les splits. Les actualités pèsent peu. Même leçon que le Rendu 3 : le calendrier explique plus que la météo ou les news.


## 9. Limites

- **Pas un retard** : pas d'horaire programmé ; le proxy `retard_minutes` n'est pas la cible.
- **Météo** : *Current* + fenêtre ± 3 h → une minorité de vols dans le jeu modèle (~17 % au 04/09).
- **Actualités** : jointure par date UTC ; un sentiment mondial par jour ne bat pas une médiane. BERT = notebook 05.
- **Chiffres** : ce notebook peut afficher un run local ou le CLI 01/09. Le livrable à citer est le **Rendu 3 (04/09)**.

Pour réécrire le joblib : `python -m src.models.entrainer`.
